# Corporate Sales Record Cleaner
## Automated Data Sanitization Pipeline

**Week 1 Internship Milestone**  
Tools: Python, Pandas, NumPy, Matplotlib (optional)

### Objective
Load a raw sales CSV, audit data quality, remove exact duplicate records, impute missing numerical/categorical values, standardize text, validate the result, and export `cleaned_sales.csv` plus summary statistics.

> This repository uses a small synthetic sample dataset for demonstration. Replace it with an authorized company dataset if your internship supervisor has supplied one. Do not upload confidential company or customer data to a public repository.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 2. Load the raw data

In [ ]:
DATA_DIR = Path(".")
input_file = DATA_DIR / "raw_sales.csv"

df = pd.read_csv(input_file)
print("First five rows:")
display(df.head())
print("Original shape (rows, columns):", df.shape)
print("\\nData types:")
print(df.dtypes)


## 3. Baseline data quality audit

In [ ]:
print("Missing values by column:")
display(df.isnull().sum().to_frame("missing_count"))

print("Exact duplicate rows:", df.duplicated().sum())

print("Numerical summary before cleaning:")
display(df.describe())


## 4. Remove exact duplicate records

In [ ]:
rows_before_dedup = len(df)
df = df.drop_duplicates().copy()
rows_after_dedup = len(df)

print(f"Rows before duplicate removal: {rows_before_dedup}")
print(f"Exact duplicate rows removed: {rows_before_dedup - rows_after_dedup}")
print(f"Rows after duplicate removal: {rows_after_dedup}")


## 5. Standardize text fields

In [ ]:
# Strip leading/trailing whitespace in text columns first.
text_columns = df.select_dtypes(include=["object", "string"]).columns
for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

# Normalize fields where capitalization should not carry meaning.
# Keep customer/product display names in title case; normalize categories consistently.
df["City"] = df["City"].str.title()
df["Payment_Method"] = df["Payment_Method"].str.lower().str.replace(r"\\s+", " ", regex=True)
df["Payment_Method"] = df["Payment_Method"].replace({
    "credit card": "credit card",
    "debit card": "debit card",
    "upi": "upi",
    "cash": "cash"
})
df["Customer"] = df["Customer"].str.title()
df["Product"] = df["Product"].str.title()

display(df.head())


## 6. Handle missing values (imputation)

In [ ]:
# Convert fields that should be numeric. Invalid entries become NaN.
numeric_columns = ["Age", "Price", "Quantity"]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Median is robust to skewed sales prices and outliers.
for col in numeric_columns:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

# Categorical columns: use mode (most frequent value) where available.
categorical_columns = ["City", "Payment_Method", "Customer", "Product"]
for col in categorical_columns:
    mode_values = df[col].mode(dropna=True)
    fill_value = mode_values.iloc[0] if not mode_values.empty else "Unknown"
    df[col] = df[col].fillna(fill_value)

# Dates are parsed consistently; any invalid date can be flagged as missing.
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
print("Missing values after imputation:")
display(df.isnull().sum().to_frame("missing_count"))


## 7. Validate cleaned data

In [ ]:
print("Cleaned shape (rows, columns):", df.shape)
print("Remaining exact duplicates:", df.duplicated().sum())
print("Total missing cells:", int(df.isnull().sum().sum()))

assert df.duplicated().sum() == 0, "Duplicate rows remain."
assert int(df.isnull().sum().sum()) == 0, "Missing values remain."

# Business-rule checks
assert (df["Price"] >= 0).all(), "Negative prices found."
assert (df["Quantity"] >= 0).all(), "Negative quantities found."

print("Validation passed.")


## 8. Export cleaned data and summary statistics

In [ ]:
# Save dates in a readable CSV format.
df["Order_Date"] = df["Order_Date"].dt.strftime("%Y-%m-%d")

output_file = DATA_DIR / "cleaned_sales.csv"
df.to_csv(output_file, index=False)

summary = df.describe(include="all").transpose()
summary_file = DATA_DIR / "summary_statistics.csv"
summary.to_csv(summary_file)

print(f"Cleaned dataset saved to: {output_file}")
print(f"Summary statistics saved to: {summary_file}")
display(summary)


## 9. Before/after comparison

In [ ]:
raw_df = pd.read_csv(DATA_DIR / "raw_sales.csv")
comparison = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Missing cells", "Exact duplicate rows"],
    "Before cleaning": [
        len(raw_df),
        raw_df.shape[1],
        int(raw_df.isnull().sum().sum()),
        int(raw_df.duplicated().sum())
    ],
    "After cleaning": [
        len(df),
        df.shape[1],
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum())
    ]
})
display(comparison)


## 10. Conclusion
The pipeline demonstrates CSV ingestion, quality auditing, exact duplicate removal, numeric median imputation, categorical mode imputation, text normalization, validation, and CSV export.

### Important interpretation notes
- Exact duplicates are removed using all columns. Similar-looking records are not automatically removed unless all values match.
- Median imputation was selected for numerical fields because sales values can be skewed by high-priced products.
- Mode imputation fills missing categorical fields with the most frequent category.
- In a real project, decide whether fields such as customer names should be imputed or marked `Unknown` based on business meaning. Do not silently invent customer data.
- This is a data-cleaning demonstration, not a production pipeline for confidential customer data.
